# Mini Trabalho 4 — Preparação dos Dados
**Universidade de Brasília — UnB | Aprendizado de Máquina**

**Grupo 5:**
- Gustavo Costa de Jesus — 211061814
- Harleny Angéllica Araújo de Sousa — 211061832
- João Pedro Araújo de Freitas Lyra — 232003661
- Raquel Ferreira Andrade — 211062437


## 1. Impots e Configuração


In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import os



## 2. Carregamento dos Dados


In [2]:
ratings = pd.read_csv('ml-latest-small/ratings.csv')
movies = pd.read_csv('ml-latest-small/movies.csv')


## 3. Limpeza e Filtragem por frequência 

In [3]:
avaliacoes_por_usuario = ratings.groupby('userId')['rating'].count()
avaliacoes_por_filme = ratings.groupby('movieId')['rating'].count()

print(f'Usuários com menos de 5 avaliações: {(avaliacoes_por_usuario < 5).sum()}')
print(f'Filmes com menos de 5 avaliações: {(avaliacoes_por_filme < 5).sum()}')



Usuários com menos de 5 avaliações: 0
Filmes com menos de 5 avaliações: 6074


In [4]:
filmes_validos = avaliacoes_por_filme[avaliacoes_por_filme >= 5].index
usuarios_validos = avaliacoes_por_usuario[avaliacoes_por_usuario >= 5].index

ratings_limpo = ratings[
                    ratings['movieId'].isin(filmes_validos) &
                    ratings['userId'].isin(usuarios_validos)
                    ]

print(f'Ratings original: {len(ratings)}')
print(f'Ratings limpo: {len(ratings_limpo)}')

Ratings original: 100836
Ratings limpo: 90274


In [5]:
print(ratings_limpo['rating'].describe())

count    90274.000000
mean         3.537358
std          1.029858
min          0.500000
25%          3.000000
50%          3.500000
75%          4.000000
max          5.000000
Name: rating, dtype: float64


## 4. Codificação dos Gêneros

In [6]:
movies_generos = movies[movies['movieId'].isin(filmes_validos)].copy()
movies_generos = movies_generos[movies_generos['genres'] != '(no genres listed)']

generos_dummies = movies_generos['genres'].str.get_dummies(sep='|')

movies_encoded = pd.concat([movies_generos[['movieId', 'title']], generos_dummies], axis=1)

print(movies_encoded.shape)
print(movies_encoded.head())

(3649, 21)
   movieId                               title  Action  Adventure  Animation  \
0        1                    Toy Story (1995)       0          1          1   
1        2                      Jumanji (1995)       0          1          0   
2        3             Grumpier Old Men (1995)       0          0          0   
3        4            Waiting to Exhale (1995)       0          0          0   
4        5  Father of the Bride Part II (1995)       0          0          0   

   Children  Comedy  Crime  Documentary  Drama  ...  Film-Noir  Horror  IMAX  \
0         1       1      0            0      0  ...          0       0     0   
1         1       0      0            0      0  ...          0       0     0   
2         0       1      0            0      0  ...          0       0     0   
3         0       1      0            0      1  ...          0       0     0   
4         0       1      0            0      0  ...          0       0     0   

   Musical  Mystery  Romanc

## 5. Normalização das Notas

In [7]:
media_por_usuario = ratings_limpo.groupby('userId')['rating'].mean()

ratings_normalizado = ratings_limpo.copy()
ratings_normalizado['rating_norm'] = ratings_normalizado.apply(
    lambda row: row['rating'] - media_por_usuario[row['userId']], axis=1
)

print(ratings_normalizado[['userId', 'movieId', 'rating', 'rating_norm']].head(10))

   userId  movieId  rating  rating_norm
0       1        1     4.0    -0.361233
1       1        3     4.0    -0.361233
2       1        6     4.0    -0.361233
3       1       47     5.0     0.638767
4       1       50     5.0     0.638767
5       1       70     3.0    -1.361233
6       1      101     5.0     0.638767
7       1      110     4.0    -0.361233
8       1      151     5.0     0.638767
9       1      157     5.0     0.638767


In [8]:
scaler = MinMaxScaler()

ratings_normalizado['rating_scaled'] = scaler.fit_transform(
    ratings_normalizado[['rating_norm']]
)

print(ratings_normalizado[['userId', 'movieId', 'rating', 'rating_norm', 'rating_scaled']].head(10))

   userId  movieId  rating  rating_norm  rating_scaled
0       1        1     4.0    -0.361233       0.484795
1       1        3     4.0    -0.361233       0.484795
2       1        6     4.0    -0.361233       0.484795
3       1       47     5.0     0.638767       0.617611
4       1       50     5.0     0.638767       0.617611
5       1       70     3.0    -1.361233       0.351979
6       1      101     5.0     0.638767       0.617611
7       1      110     4.0    -0.361233       0.484795
8       1      151     5.0     0.638767       0.617611
9       1      157     5.0     0.638767       0.617611


## 6. Exportação dos Dados Processados

In [9]:
os.makedirs('dados_processados', exist_ok=True)
ratings_normalizado.to_csv('dados_processados/ratings_processado.csv', index=False)
movies_encoded.to_csv('dados_processados/movies_processado.csv', index=False)

print('Arquivos salvos!')

Arquivos salvos!


## 7. Conclusões

A partir dos dados explorados no Mini Trabalho 3, realizamos as seguintes etapas de preparação:

Na etapa de limpeza, filtramos filmes com menos de 5 avaliações, removendo 10.562 registros do dataset. Não havia usuários com menos de 5 avaliações, nem valores nulos ou duplicatas, conforme já identificado anteriormente.

Em seguida, aplicamos binarização nos gêneros, transformando a coluna `genres` em colunas binárias — uma por gênero — permitindo que o modelo interprete essa informação de forma numérica.

Por fim, normalizamos as notas em duas etapas: primeiro subtraindo a média individual de cada usuário para corrigir o viés pessoal (`rating_norm`), e depois aplicando MinMaxScaler para escalar todos os valores entre 0 e 1 (`rating_scaled`).